# SPAR · Headless Quickstart (no GUI required)

This notebook scores 2,000 short documents on two theoretical concepts in
under one minute, using only the headless `score()` API. No web UI, no
manual seed iteration: ideal for batch jobs, reproducible pipelines, and
Colab demos.

For the **interactive GUI version** (with active retrieval and live seed
refinement), open the companion notebook:
[example_colab.ipynb](https://colab.research.google.com/github/maifeng/SPAR_measure/blob/master/resources/example_colab.ipynb).

Originally developed for:
Yan, Bei, Feng Mai, Chaojiang Wu, Rong Chen, and Xiaolin Li (2024).
"A Computational Framework for Understanding Firm Communication
During Disasters." *Information Systems Research* 35(2):590-608.
https://doi.org/10.1287/isre.2022.0128

## 1. Install

In [ ]:
!pip install -q -U spar-measure

## 2. Load the bundled sample corpus

`spar_measure` ships with 2,000 Russell-3000 Facebook posts from the
ISR paper, plus pre-computed sentence embeddings so you do not have to
wait for the embedding pass on first run.

In [ ]:
from importlib.resources import files
import numpy as np
import pandas as pd

sample = files("spar_measure.sample_data")
docs = pd.read_csv(sample / "sample_text.csv")
embeddings = np.load(sample / "sample_emb.npy")

print(f"Corpus: {len(docs)} documents")
print(f"Embeddings: shape {embeddings.shape}")
docs.head(3)

## 3. Define dimensions and scales

A *dimension* is a concept defined by one or more seed sentences.
A *scale* combines positive and negative dimensions into a signed
bipolar construct. Below we define four CVF poles (Create, Compete,
Collaborate, Control) and combine them into the two classic CVF axes:
External-Internal and Flexible-Stable.

In [ ]:
scales = {
    "dimensions": {
        "Create":      {"queries": ["We should adapt and innovate.",
                                    "Creativity and agility define how we work."]},
        "Compete":     {"queries": ["We push to outperform the competition.",
                                    "Winning market share is the top priority."]},
        "Collaborate": {"queries": ["We value teamwork and mutual support.",
                                    "Our strength comes from working together."]},
        "Control":     {"queries": ["Strict procedures keep things running smoothly.",
                                    "Consistency and compliance matter most here."]},
    },
    "scales": {
        "External-Internal": {"pos_dims": ["Create", "Compete"],
                              "neg_dims": ["Collaborate", "Control"]},
        "Flexible-Stable":   {"pos_dims": ["Create", "Collaborate"],
                              "neg_dims": ["Compete", "Control"]},
    },
}

## 4. Score in one call

`precomputed_embeddings=` lets us reuse the bundled embedding matrix.
Drop that argument and `score()` will embed the corpus from scratch
with `all-MiniLM-L6-v2` (about 30 s on a Colab T4 for 2,000 docs).

In [ ]:
from spar_measure import score

out = score(
    docs,
    scales,
    text_col="text",
    id_col="doc_id",
    precomputed_embeddings=embeddings,
)
out.head(10)

## 5. Inspect the score distribution

In [ ]:
out[["External-Internal", "Flexible-Stable"]].describe()

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, col in zip(axes, ["External-Internal", "Flexible-Stable"]):
    ax.hist(out[col], bins=40, edgecolor="white", alpha=0.85, color="#9E1B32")
    ax.set_title(col)
    ax.set_xlabel("Score")
fig.suptitle("SPAR scores on 2,000 Russell-3000 Facebook posts", fontsize=13)
fig.tight_layout()
plt.show()

## 6. Decorrelate scales with ZCA whitening

When two scales share dimensions (here, both include `Create`) their
scores are correlated by construction. ZCA whitening rotates them to
be orthogonal while preserving the geometry as much as possible.
Useful when you plan to use the scores as regressors.

In [ ]:
out_whitened = score(
    docs, scales,
    text_col="text", id_col="doc_id",
    precomputed_embeddings=embeddings,
    whiten=True,
)
print("Correlation (raw) :",
      out[["External-Internal", "Flexible-Stable"]].corr().iloc[0, 1].round(3))
print("Correlation (ZCA) :",
      out_whitened[["External-Internal", "Flexible-Stable"]].corr().iloc[0, 1].round(3))

## 7. Bring your own construct

SPAR is not limited to CVF. Any construct that can be expressed as a
few seed sentences works: ESG concern, customer focus, risk tolerance,
political stance, you name it. Below: a simple "people vs performance"
scale.

In [ ]:
custom_scales = {
    "dimensions": {
        "People":      {"queries": ["We care about our employees and their well-being.",
                                    "The company invests in people, not just profits."]},
        "Performance": {"queries": ["Results are what matter here.",
                                    "We measure everything and hold people accountable."]},
    },
    "scales": {
        "People-Performance": {"pos_dims": ["People"], "neg_dims": ["Performance"]},
    },
}

out_custom = score(
    docs, custom_scales,
    text_col="text", id_col="doc_id",
    precomputed_embeddings=embeddings,
)
out_custom.head(10)

## 8. Switch to OpenAI embeddings

To use OpenAI's hosted embedding model instead of the local
Sentence-BERT model, pass `openai_api_key=`. SPAR will re-embed the
corpus through the API. Costs about \$0.02 per 1,000 short documents
at current `text-embedding-3-small` prices.

In [ ]:
# import os
# os.environ["OPENAI_API_KEY"] = "sk-..."
# out_openai = score(
#     docs, scales,
#     text_col="text", id_col="doc_id",
#     openai_api_key=os.environ["OPENAI_API_KEY"],
#     openai_embedding_model="text-embedding-3-small",
# )

## 9. Want to iterate seeds interactively?

Open the GUI companion notebook for the full active-retrieval workflow
(search the corpus for exemplar sentences, refine seeds, re-score):

[![Open GUI in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maifeng/SPAR_measure/blob/master/resources/example_colab.ipynb)

Or launch the GUI locally:

```bash
python -m spar_measure gui
```

## 10. Citation

```
Yan, Bei, Feng Mai, Chaojiang Wu, Rong Chen, and Xiaolin Li. 2024.
"A Computational Framework for Understanding Firm Communication
During Disasters."
Information Systems Research 35(2):590-608.
https://doi.org/10.1287/isre.2022.0128
```

In [ ]:
import spar_measure
print(spar_measure.__paper__)